# CSE422 Lab Project — Mushroom Edibility Prediction

This notebook is Colab-ready. Upload `mushroom_dataset.xlsx` into the working directory before running.

**Two run modes**:
- `SUBSAMPLE = True` (fast; uses a small subset for quick demo)
- `SUBSAMPLE = False` (runs on full dataset; may require more RAM/time)


In [ ]:
SUBSAMPLE = True  # set False to run on full dataset
SUBSAMPLE_SIZE = 3000
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from pathlib import Path
DATA_PATH = Path('mushroom_dataset.xlsx')
if not DATA_PATH.exists():
    raise FileNotFoundError('Please upload mushroom_dataset.xlsx to the notebook working directory')
df = pd.read_excel(DATA_PATH)
print('Loaded dataset with shape:', df.shape)


In [ ]:
df.info()
display(df.head())
print('Top null counts:')
print(df.isnull().sum().sort_values(ascending=False).head(20))

In [ ]:
if 'cap-diameter' in df.columns:
    df['cap_size_band'] = pd.cut(df['cap-diameter'], bins=[-np.inf,2,5,10,np.inf], labels=['tiny','small','medium','large'])
else:
    df['cap_size_band'] = 'unknown'
if set(['cap-surface', 'cap-color']).issubset(df.columns):
    df['cap_surface_color'] = df['cap-surface'].astype(str) + '_' + df['cap-color'].astype(str)
else:
    df['cap_surface_color'] = 'unknown'
if set(['stem-height','stem-width']).issubset(df.columns):
    df['stem_hw_ratio'] = df['stem-height'] / (df['stem-width'].replace(0, np.nan))
else:
    df['stem_hw_ratio'] = np.nan
df.loc[df.sample(frac=0.02, random_state=7).index, 'cap_size_band'] = np.nan
df.loc[df.sample(frac=0.02, random_state=8).index, 'cap_surface_color'] = np.nan
print('Engineered features added.')


In [ ]:
target_col = 'class'
y_raw = df[target_col].astype(str).str.lower().str.strip()
label_map = {'e': 0, 'edible': 0, 'safe': 0, '0': 0, 'p': 1, 'poisonous': 1, 'toxic': 1, '1': 1}
y = y_raw.map(label_map)
if y.isna().any():
    y = pd.factorize(y_raw)[0]
df = df.drop_duplicates().reset_index(drop=True)
X = df.drop(columns=[target_col])
numeric_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
categorical_cols = [c for c in X.columns if c not in numeric_cols]
X[numeric_cols] = X[numeric_cols].apply(lambda s: s.fillna(s.median()))
X[categorical_cols] = X[categorical_cols].astype(str).replace(['nan','None','NaN'], np.nan).fillna('Unknown')
print('Prepared X with shape', X.shape)


In [ ]:
from sklearn.model_selection import train_test_split
if SUBSAMPLE:
    subsample_size = min(SUBSAMPLE_SIZE, len(X))
    X_sub, _, y_sub, _ = train_test_split(X, y.loc[X.index], train_size=subsample_size, stratify=y.loc[X.index], random_state=42)
else:
    X_sub = X.copy(); y_sub = y.loc[X.index]
print('Working on', len(X_sub), 'rows')
X_train, X_test, y_train, y_test = train_test_split(X_sub, y_sub, test_size=0.2, stratify=y_sub, random_state=42)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, roc_auc_score, roc_curve
preprocess = ColumnTransformer(transformers=[
    ('num', StandardScaler(with_mean=False), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])
models = {
    'LogisticRegression': LogisticRegression(max_iter=500),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'BernoulliNB': BernoulliNB(),
    'MLP': MLPClassifier(hidden_layer_sizes=(32,), max_iter=200, random_state=42)
}
results = []
preds = {}
probas = {}
for name, clf in models.items():
    pipe = Pipeline([('pre', preprocess), ('m', clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    if hasattr(pipe.named_steps['m'], 'predict_proba'):
        y_score = pipe.predict_proba(X_test)[:,1]
    else:
        try:
            y_score = pipe.decision_function(X_test)
            y_score = (y_score - y_score.min()) / (y_score.max() - y_score.min())
        except Exception:
            y_score = y_pred.astype(float)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_score)
    cm = confusion_matrix(y_test, y_pred)
    results.append({'model':name,'accuracy':acc,'f1':f1,'precision':prec,'recall':rec,'auc':auc,'tn':cm[0,0],'fp':cm[0,1],'fn':cm[1,0],'tp':cm[1,1]})
    preds[name]=y_pred; probas[name]=y_score
import pandas as pd
results_df = pd.DataFrame(results).sort_values(by='f1', ascending=False).reset_index(drop=True)
display(results_df)
results_df.to_csv('model_performance_notebook.csv', index=False)


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
plt.bar(results_df['model'], results_df['accuracy'])
plt.title('Accuracy by model'); plt.xlabel('Model'); plt.ylabel('Accuracy')
plt.tight_layout(); plt.savefig('accuracy_bar.png')
plt.show()
plt.figure(figsize=(6,4))
plt.bar(results_df['model'], results_df['f1'])
plt.title('F1 by model'); plt.xlabel('Model'); plt.ylabel('F1')
plt.tight_layout(); plt.savefig('f1_bar.png')
plt.show()
best = results_df.loc[0,'model']
cm = confusion_matrix(y_test, preds[best])
plt.figure(figsize=(4,3)); plt.imshow(cm, interpolation='nearest')
plt.title(f'Confusion Matrix - {best}'); plt.colorbar(); plt.tight_layout(); plt.savefig(f'confusion_matrix_{best}.png')
plt.show()
fpr, tpr, _ = roc_curve(y_test, probas[best])
plt.figure(figsize=(5,4)); plt.plot(fpr, tpr, label=f'AUC={results_df.loc[0,"auc"]:.3f}'); plt.plot([0,1],[0,1],'--'); plt.legend(); plt.title('ROC'); plt.tight_layout(); plt.savefig(f'roc_{best}.png'); plt.show()


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, homogeneity_score
X_tr = preprocess.fit_transform(X_sub)
km = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters = km.fit_predict(X_tr)
print('ARI:', adjusted_rand_score(y_sub, clusters))
print('Homogeneity:', homogeneity_score(y_sub, clusters))
pd.DataFrame({'cluster': clusters}).to_csv('kmeans_clusters_notebook.csv', index=False)
